# Data Exploration - Respira SP

Notebook inicial para explorar os datasets de poluição do ar e clima.

In [41]:
!pip install requests-cache retry-requests openmeteo-requests

  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.5/772.5 kB 12.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 17.7 MB/s  0:00:00
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.2.10
    Uninstalling flatbuffers-25.2.10:
      Successfully uninstalled flatbuffers-25.2.10
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Uninstalling typing_extensions-4.12.2:
      Successfully uninstalled typing_extensions-4.12.2━━━━━━━━━━━  3/14 [typing-extensions]
  Attempting uninstall: attrs0m╺━━━━━━━━━━━━━━━━━━━━━━━━━  5/14 [openmeteo-sdk]ons]
    Found existing installation: attrs 25.1.0━━━━━━━━━━━━━━━━━  5/14 [openmeteo-sdk]
    Uninstall

In [47]:
import pandas as pd
import numpy as np
from pathlib import Path
import kagglehub
import requests_cache
from retry_requests import retry
import openmeteo_requests
pd.set_option('display.max_columns', None)

In [5]:

path = kagglehub.dataset_download("danlessa/air-pollution-at-so-paulo-brazil-since-2013")

100%|███████████████████████████████████████████████████████████████████████████████████████████| 85.4M/85.4M [00:04<00:00, 22.1MB/s]

Extracting files...


In [7]:
dataset_path = Path(path)

list(dataset_path.rglob("*"))

[PosixPath('/home/markz/.cache/kagglehub/datasets/danlessa/air-pollution-at-so-paulo-brazil-since-2013/versions/1/cetesb.csv.gz'),
 PosixPath('/home/markz/.cache/kagglehub/datasets/danlessa/air-pollution-at-so-paulo-brazil-since-2013/versions/1/estacoes-cetesb.csv'),
 PosixPath('/home/markz/.cache/kagglehub/datasets/danlessa/air-pollution-at-so-paulo-brazil-since-2013/versions/1/cetesb.csv.gzip'),
 PosixPath('/home/markz/.cache/kagglehub/datasets/danlessa/air-pollution-at-so-paulo-brazil-since-2013/versions/1/cetesb.csv'),
 PosixPath('/home/markz/.cache/kagglehub/datasets/danlessa/air-pollution-at-so-paulo-brazil-since-2013/versions/1/estacoes-cetesb.csv.gz'),
 PosixPath('/home/markz/.cache/kagglehub/datasets/danlessa/air-pollution-at-so-paulo-brazil-since-2013/versions/1/estacoes-cetesb.csv/estacoes-cetesb.csv'),
 PosixPath('/home/markz/.cache/kagglehub/datasets/danlessa/air-pollution-at-so-paulo-brazil-since-2013/versions/1/cetesb.csv/cetesb.csv')]

In [11]:
csv_files = [file for file in dataset_path.rglob("*.csv") if file.is_file()]

csv_files

[PosixPath('/home/markz/.cache/kagglehub/datasets/danlessa/air-pollution-at-so-paulo-brazil-since-2013/versions/1/estacoes-cetesb.csv/estacoes-cetesb.csv'),
 PosixPath('/home/markz/.cache/kagglehub/datasets/danlessa/air-pollution-at-so-paulo-brazil-since-2013/versions/1/cetesb.csv/cetesb.csv')]

In [12]:
estacoes_file = [file for file in csv_files if "estacoes" in file.name.lower()][0]

estacoes = pd.read_csv(estacoes_file)

estacoes.head()

,DATA,Efeito,Endereco,ID,Indice,MsgSaude,Municipio,Nome,POLUENTE,Qualidade,Situacao_Rede,Tipo_Rede,x,y,ID2
0,2019-08-18 18:00:00.000,Pessoas com doenças respiratórias podem aprese...,"R TAILANDIA, 364",1,43.0,"Pessoas com doenças cardíacas ou pulmonares, p...",AMERICANA,Americana,MP10,N2 - MODERADA,A,A,-47.347456,-22.724507,65
1,2019-08-18 18:00:00.000,NaN,"R CLOVIS PESTANA, 801",2,34.0,NaN,ARACATUBA,Araçatuba,MP10,N1 - BOA,A,A,-50.439317,-21.186841,72
2,2019-08-18 18:00:00.000,NaN,"AV ANGELO HORTENCE, 1990",3,37.0,NaN,ARARAQUARA,Araraquara,O3,N1 - BOA,A,A,-48.185832,-21.782522,71
3,2019-08-18 18:00:00.000,Pessoas com doenças respiratórias podem aprese...,"R CASTRO ALVES, s/nº",4,45.0,"Pessoas com doenças cardíacas ou pulmonares, p...",BAURU,Bauru,MP10,N2 - MODERADA,A,A,-49.092759,-22.326608,73
4,2019-08-18 18:00:00.000,NaN,"AV ANCHIETA, 42",5,30.0,NaN,CAMPINAS,Campinas-Centro,MP10,N1 - BOA,A,A,-47.057211,-22.902525,42


In [13]:
cetesb_file = [file for file in csv_files if file.name.lower() == "cetesb.csv"][0]

cetesb = pd.read_csv(cetesb_file)

cetesb.head()

,Unnamed: 0,time,id,MP10,TRS,O3,NO2,CO,MP2.5,SO2,BENZENO,TOLUENO
0,0,2013-05-09 00:00:00,65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2013-05-09 01:00:00,65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,2013-05-09 02:00:00,65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2013-05-09 03:00:00,65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2013-05-09 04:00:00,65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
cetesb.isnull().sum().sort_values(ascending=False)

TRS           3310360
TOLUENO       3241320
BENZENO       3241113
SO2           2702465
MP2.5         2549289
CO            2477907
NO2           1689231
O3            1163498
MP10           972030
Unnamed: 0          0
time                0
id                  0
dtype: int64

In [16]:
cetesb.isnull().sum().sort_values(ascending=False) / len(cetesb)

TRS           0.986572
TOLUENO       0.965996
BENZENO       0.965935
SO2           0.805404
MP2.5         0.759753
CO            0.738480
NO2           0.503434
O3            0.346752
MP10          0.289690
Unnamed: 0    0.000000
time          0.000000
id            0.000000
dtype: float64

In [18]:
cetesb.shape

(3355416, 12)

In [22]:
cetesb['id'].head(10)

0    65
1    65
2    65
3    65
4    65
5    65
6    65
7    65
8    65
9    65
Name: id, dtype: int64

In [28]:
cetesb_id_8 = cetesb[cetesb["id"] == 8]

In [29]:
missing_rate = cetesb_id_8["MP2.5"].isnull().sum() / len(cetesb_id_8)

In [30]:
missing_rate

0.060573007562536356

In [31]:
missing_percent = cetesb_id_8["MP2.5"].isnull().mean() * 100
missing_percent

6.057300756253635

In [34]:
cetesb_id_8

,Unnamed: 0,time,id,MP10,TRS,O3,NO2,CO,MP2.5,SO2,BENZENO,TOLUENO
288,288,2013-05-09 00:00:00,8,69.0,NaN,NaN,NaN,1.0,26.0,9.0,NaN,NaN
289,289,2013-05-09 01:00:00,8,35.0,NaN,NaN,NaN,0.6,21.0,6.0,NaN,NaN
290,290,2013-05-09 02:00:00,8,34.0,NaN,NaN,NaN,0.6,20.0,6.0,NaN,NaN
291,291,2013-05-09 03:00:00,8,35.0,NaN,NaN,NaN,0.5,18.0,NaN,NaN,NaN
292,292,2013-05-09 04:00:00,8,47.0,NaN,NaN,NaN,NaN,20.0,2.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3353323,3353323,2017-05-27 19:00:00,8,22.0,NaN,NaN,76.0,0.9,17.0,3.0,NaN,NaN
3353324,3353324,2017-05-27 20:00:00,8,30.0,NaN,NaN,80.0,0.9,24.0,3.0,NaN,NaN
3353325,3353325,2017-05-27 21:00:00,8,41.0,NaN,NaN,79.0,1.0,28.0,3.0,NaN,NaN
3353326,3353326,2017-05-27 22:00:00,8,43.0,NaN,NaN,79.0,0.9,29.0,3.0,NaN,NaN


In [35]:
df_id8_mp25 = cetesb.loc[cetesb["id"] == 8, ["MP2.5"]]

In [36]:
df_id8_mp25.head()

,MP2.5
288,26.0
289,21.0
290,20.0
291,18.0
292,20.0


In [38]:
df_id8_mp25.shape

(55008, 1)

In [48]:

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": -23.6473149,
    "longitude": -46.6643635,
    "start_date": "2013-01-01",
    "end_date": "2019-12-31",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m"
    ],
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(3).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: -23.655536651611328°N -46.7088623046875°E
Elevation: 793.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                            date  temperature_2m  relative_humidity_2m  \
0     2013-01-01 00:00:00+00:00       22.651001             89.586006   
1     2013-01-01 01:00:00+00:00       22.201000             90.384933   
2     2013-01-01 02:00:00+00:00       21.801001             90.357224   
3     2013-01-01 03:00:00+00:00       21.451000             90.053207   
4     2013-01-01 04:00:00+00:00       21.351000             88.658142   
...                         ...             ...                   ...   
61339 2019-12-31 19:00:00+00:00       26.700001             70.325050   
61340 2019-12-31 20:00:00+00:00       25.900000             76.961960   
61341 2019-12-31 21:00:00+00:00       24.000000             79.084686   
61342 2019-12-31 22:00:00+00:00       23.000000             86.899796   
61343 2019-12-31 23:00:00+00:00       22.100000             91.781471   

In [49]:
hourly_dataframe.head()

,date,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
0,2013-01-01 00:00:00+00:00,22.651001,89.586006,0.0,3.976330
1,2013-01-01 01:00:00+00:00,22.201000,90.384933,0.0,6.489992
2,2013-01-01 02:00:00+00:00,21.801001,90.357224,0.0,8.404284
3,2013-01-01 03:00:00+00:00,21.451000,90.053207,0.0,8.759178
4,2013-01-01 04:00:00+00:00,21.351000,88.658142,0.0,8.654986


In [50]:
cetesb_id_8

,Unnamed: 0,time,id,MP10,TRS,O3,NO2,CO,MP2.5,SO2,BENZENO,TOLUENO
288,288,2013-05-09 00:00:00,8,69.0,NaN,NaN,NaN,1.0,26.0,9.0,NaN,NaN
289,289,2013-05-09 01:00:00,8,35.0,NaN,NaN,NaN,0.6,21.0,6.0,NaN,NaN
290,290,2013-05-09 02:00:00,8,34.0,NaN,NaN,NaN,0.6,20.0,6.0,NaN,NaN
291,291,2013-05-09 03:00:00,8,35.0,NaN,NaN,NaN,0.5,18.0,NaN,NaN,NaN
292,292,2013-05-09 04:00:00,8,47.0,NaN,NaN,NaN,NaN,20.0,2.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3353323,3353323,2017-05-27 19:00:00,8,22.0,NaN,NaN,76.0,0.9,17.0,3.0,NaN,NaN
3353324,3353324,2017-05-27 20:00:00,8,30.0,NaN,NaN,80.0,0.9,24.0,3.0,NaN,NaN
3353325,3353325,2017-05-27 21:00:00,8,41.0,NaN,NaN,79.0,1.0,28.0,3.0,NaN,NaN
3353326,3353326,2017-05-27 22:00:00,8,43.0,NaN,NaN,79.0,0.9,29.0,3.0,NaN,NaN


In [51]:
cetesb_id_8.drop(columns=['MP10', 'TRS', 'O3', 'NO2' ])

,Unnamed: 0,time,id,CO,MP2.5,SO2,BENZENO,TOLUENO
288,288,2013-05-09 00:00:00,8,1.0,26.0,9.0,NaN,NaN
289,289,2013-05-09 01:00:00,8,0.6,21.0,6.0,NaN,NaN
290,290,2013-05-09 02:00:00,8,0.6,20.0,6.0,NaN,NaN
291,291,2013-05-09 03:00:00,8,0.5,18.0,NaN,NaN,NaN
292,292,2013-05-09 04:00:00,8,NaN,20.0,2.0,NaN,NaN
...,...,...,...,...,...,...,...,...
3353323,3353323,2017-05-27 19:00:00,8,0.9,17.0,3.0,NaN,NaN
3353324,3353324,2017-05-27 20:00:00,8,0.9,24.0,3.0,NaN,NaN
3353325,3353325,2017-05-27 21:00:00,8,1.0,28.0,3.0,NaN,NaN
3353326,3353326,2017-05-27 22:00:00,8,0.9,29.0,3.0,NaN,NaN


In [53]:
cetesb_id_8.drop(columns=['CO', 'SO2', 'BENZENO' , 'TOLUENO' ])

,Unnamed: 0,time,id,MP10,TRS,O3,NO2,MP2.5
288,288,2013-05-09 00:00:00,8,69.0,NaN,NaN,NaN,26.0
289,289,2013-05-09 01:00:00,8,35.0,NaN,NaN,NaN,21.0
290,290,2013-05-09 02:00:00,8,34.0,NaN,NaN,NaN,20.0
291,291,2013-05-09 03:00:00,8,35.0,NaN,NaN,NaN,18.0
292,292,2013-05-09 04:00:00,8,47.0,NaN,NaN,NaN,20.0
...,...,...,...,...,...,...,...,...
3353323,3353323,2017-05-27 19:00:00,8,22.0,NaN,NaN,76.0,17.0
3353324,3353324,2017-05-27 20:00:00,8,30.0,NaN,NaN,80.0,24.0
3353325,3353325,2017-05-27 21:00:00,8,41.0,NaN,NaN,79.0,28.0
3353326,3353326,2017-05-27 22:00:00,8,43.0,NaN,NaN,79.0,29.0


In [55]:
cetesb_id_8 = cetesb_id_8.drop(columns=['CO', 'SO2', 'BENZENO' , 'TOLUENO', 'MP10', 'TRS', 'O3', 'NO2'])

In [56]:
cetesb_id_8

,Unnamed: 0,time,id,MP2.5
288,288,2013-05-09 00:00:00,8,26.0
289,289,2013-05-09 01:00:00,8,21.0
290,290,2013-05-09 02:00:00,8,20.0
291,291,2013-05-09 03:00:00,8,18.0
292,292,2013-05-09 04:00:00,8,20.0
...,...,...,...,...
3353323,3353323,2017-05-27 19:00:00,8,17.0
3353324,3353324,2017-05-27 20:00:00,8,24.0
3353325,3353325,2017-05-27 21:00:00,8,28.0
3353326,3353326,2017-05-27 22:00:00,8,29.0


In [57]:
hourly_dataframe

,date,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
0,2013-01-01 00:00:00+00:00,22.651001,89.586006,0.0,3.976330
1,2013-01-01 01:00:00+00:00,22.201000,90.384933,0.0,6.489992
2,2013-01-01 02:00:00+00:00,21.801001,90.357224,0.0,8.404284
3,2013-01-01 03:00:00+00:00,21.451000,90.053207,0.0,8.759178
4,2013-01-01 04:00:00+00:00,21.351000,88.658142,0.0,8.654986
...,...,...,...,...,...
61339,2019-12-31 19:00:00+00:00,26.700001,70.325050,0.3,15.021105
61340,2019-12-31 20:00:00+00:00,25.900000,76.961960,0.0,18.075441
61341,2019-12-31 21:00:00+00:00,24.000000,79.084686,0.0,17.566378
61342,2019-12-31 22:00:00+00:00,23.000000,86.899796,0.0,11.212135


In [58]:
cetesb_id_8["time"] = pd.to_datetime(cetesb_id_8["time"])
hourly_dataframe["date"] = pd.to_datetime(hourly_dataframe["date"]).dt.tz_localize(None)

df_merged = pd.merge(
    cetesb_id_8,
    hourly_dataframe,
    left_on="time",
    right_on="date",
    how="inner"
)

df_merged = df_merged.drop(columns=["date", "Unnamed: 0"])

df_merged = df_merged.reset_index(drop=True)

df_merged.head()

,time,id,MP2.5,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
0,2013-05-09 00:00:00,8,26.0,12.001000,86.403580,0.0,5.815978
1,2013-05-09 01:00:00,8,21.0,11.201000,89.879234,0.0,5.937272
2,2013-05-09 02:00:00,8,20.0,10.701000,91.365921,0.0,7.200000
3,2013-05-09 03:00:00,8,18.0,10.301001,91.648323,0.0,7.421590
4,2013-05-09 04:00:00,8,20.0,9.751000,92.548309,0.0,7.993298


In [59]:
df_merged.shape

(55008, 7)